## **Libraries & Data Loading**

**Loading Libraries**

In [ ]:
import numpy as np
import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
import plotly.express as px
import os

import warnings
warnings.filterwarnings("ignore")

Agar code se bhi blank screen aaye, toh **pio.renderers.default = 'notebook_connected'** ki line ko hata kar **pio.renderers.default = 'iframe'** ya **'vscode'** try karein. iframe har dafa kaam kar jata hai kyunki yeh graph ko ek alag isolated block mein render karta hai

In [ ]:
import plotly.io as pio

# Telling Plotly to render inside vscode
pio.renderers.default = 'vscode'


**2. Loading Dataset By Kaggle CLI**

In [ ]:
# !kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/dataset --unzip

In [ ]:
# base_path = "/content/dataset/retail_contaminated_dataset"  # ya retail_clean_dataset
# files = os.listdir(base_path)
# print("Files:", files)

# csv_files = [f for f in files if f.endswith(".csv")]

# df = {}
# for file in csv_files:
#     file_path = os.path.join(base_path, file)
#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} (Shape: {df[df_name].shape})")

**3. Loading Dataset By Local Memory**

In [ ]:
# import  zipfile

# base_path = r"Datasets\retail_contaminated_dataset.zip"

# with zipfile.ZipFile(base_path, 'r' ) as zip_ref:
#     zip_ref.extractall(r"Datasets\extracted_files")


files = os.listdir(r"Datasets\extracted_files")
print("Files:", files)

csv_files = [f for f in files if f.endswith(".csv")]

df = {}
for file in csv_files:
    file_path = os.path.join(r"Datasets\extracted_files", file)
    df_name = file.replace('.csv', '')

    if df_name == "sales_transactions":
        continue  # Skip loading the sales_transactions.csv file
    
    df[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file} (Shape: {df[df_name].shape})")

**Loading `processed_sales_transactions` File**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
df['processed_sales_transactions'] = pd.read_parquet(r"Datasets/extracted_files/processed_sales_transactions.parquet")

# df['processed_sales_transactions'] = pd.read_parquet(r"/content/drive/MyDrive/Zidio/processed_sales_transactions.parquet")

In [ ]:
df_sales = df['processed_sales_transactions'].iloc[:3000000]

df_sales.shape

In [ ]:
df_sales.head()

## **Day 04 - Exploratory Data Analysis**

### **Sales Performance & Trend Analysis (Time-Series)**

In [ ]:
grouped_data= df['processed_sales_transactions'].groupby(['year', 'month', 'quarter','day'])['total_value'].sum().reset_index()

In [ ]:
grouped_data

**By Reveneue**

In [ ]:
px.bar(grouped_data, x='month', y='total_value', animation_frame='year', title='Sales Analysis by Year', labels={'total_value': 'Total Revenue', 'month': 'Month'},
       color='total_value', color_continuous_scale='emrld')

**By Quarter**

In [ ]:
px.bar(grouped_data, x='month', y='total_value', animation_frame='year', title='Sales Analysis by Year', labels={'total_value': 'Total Revenue', 'month': 'Month'},
       color='quarter', color_continuous_scale='emrld')

**Monthly Revenue & Profit (side by side)**

only revenue can't tell about which month we get most profit. Therefore i'm taking profit and revenue side by side

In [ ]:
monthly_data = df['processed_sales_transactions'].groupby(['year','month'], observed=True).agg( revenue = ('total_value', 'sum'), profit = ('total_profit', 'sum')).reset_index()

In [ ]:
px.line( monthly_data, x='month', y=['revenue', 'profit'], title='Monthly Revenue vs Profit by Year', labels={'value': 'Amount', 'month': 'Month'},
         color='year')


### **Customer & Store Behavior Analysis**

**Top 10 Stores by Revenue and Profit**

In [ ]:
stores_data = df['processed_sales_transactions'].groupby('store_id').agg(total_value = ('total_value', 'sum'), total_profit = ('total_profit', 'sum')).reset_index()

In [ ]:
stores_data.head()

In [ ]:
top_stores = df['store_master'].merge(stores_data, on='store_id').sort_values(by='total_value', ascending=False).head(10)
top_stores

In [ ]:
px.bar(
    top_stores,
    x='store_name',
    y='total_value',
    title='Top 10 Stores by Revenue (colored by Profit)',
    labels={'total_value': 'Total Revenue', 'store_name': 'Store Name'},
    color='total_profit',
    color_continuous_scale='emrld'
)

**Quantity per Order**

In [ ]:
df['processed_sales_transactions'].head()

In [ ]:
px.histogram(df['processed_sales_transactions'], x='quantity', labels={'quantity': 'Quantity'}, title='Basket Size / Quantity per Order', color='quantity')

**Quantity vs. Total Value**

In [ ]:
px.scatter(df['processed_sales_transactions'], x='quantity',y='total_value', labels={'quantity': 'Quantity', 'total_value' : 'Total Value'}, title='Quantity vs Total Value')

### **Product & Inventory Insights**

**Top 10 Selling SKUs — by quantity and by profit**

> Using Mini Dataset `df_sales` to reduce power consumption

In [ ]:
skus_data = df_sales.groupby('sku_id').agg(
    quantity = ('quantity', 'sum'),
    total_value = ('total_value', 'sum'),
    total_profit = ('total_profit', 'sum')
).reset_index()


In [ ]:
top_skus= df['sku_master'].merge(skus_data, on='sku_id')

In [ ]:
top_skus_by_qnty=top_skus.sort_values(by='quantity', ascending=False).head(10)
top_skus_by_qnty

In [ ]:
px.bar(
    top_skus_by_qnty,
    x='quantity',
    y='sku_name',
    title='Top 10 Selling SKUs by Quantity (colored by Profit)',
    labels={'quantity': 'Quantity', 'sku_name': 'SKU Name'},
    color='total_profit',
    color_continuous_scale='emrld'
)

Best sellers by quantity aren't always the most profitable.
Therefore we check second way:

In [ ]:
top_skus_by_pft = top_skus.sort_values(by='total_profit', ascending=False).head(10)

px.bar(
    top_skus_by_pft,
    x='total_profit',
    y='sku_name',
    title='Top 10 SKUs by Profit',
    labels={'total_profit': 'Total Profit', 'sku_name': 'SKU Name'},
    color='quantity',
    color_continuous_scale='emrld'
)

**Channel-wise Sales Distribution**

In [ ]:
px.pie(df_sales, names='channel', values='total_value', title='Channel-wise Sales Distribution', hole=0.4)

**Category / subcategory revenue treemap**

In [ ]:
treemap_data = df_sales.groupby(['channel', 'category', 'subcategory'])['total_value'].sum().reset_index()

In [ ]:
px.treemap(
    treemap_data,
    path=['channel', 'category', 'subcategory'],
    values='total_value',
    title='Channel > Category > Subcategory Revenue',
    color='total_value',
)

### **Discount & Promotion Impact Analysis**

In [ ]:
df_sales.head()

In [ ]:
df_sales.info()

In [ ]:
px.histogram(df['processed_sales_transactions'], x='promo_applied', title='Sales Transactions: Promo vs No Promo', color='promo_applied')

Why only True?

In [ ]:
df['processed_sales_transactions'].head()

In [ ]:
df['processed_sales_transactions']['promo_applied'].value_counts()

Bcuz i use only few million records instead of all 10million due to computational efficiency.

- Agar aapko sirf 3 Million random records chahiye computation bachane ke liye:
df_sampled = df_original.sample(n=3000000, random_state=42)

- Phir is sampled data par value_counts check karein, dono values wapas aa jayengi:
print(df_sampled['promo_applied'].value_counts())

**Discount Percentage vs. Quantity vs. Profit** 
 
> Bigger discount causes more sales but at what cost w.r.t margin?

In [ ]:
px.scatter(
    df_sales,
    x='discount_pct', y='quantity', color='total_profit',
    title='Discount % vs Quantity (colored by total profit)',
    color_continuous_scale='RdYlGn'
)

### **Interdependency of Sales Features**

In [ ]:
corr = df_sales[['quantity', 'unit_price', 'total_value', 'discount_pct', 'total_profit']].corr().round(2)

In [ ]:
px.density_heatmap(
    corr,
    title="Interdependency of Sales Features",
    text_auto=True,
    color_continuous_scale='emrld'
)

### **Region With Most Sales**

In [ ]:
df_sales().head()

In [ ]:
aggregated_sales = df['sales_transactions'].merge(df['store_master'], on='store_id')
region = aggregated_sales.groupby('city').agg(
    total_value=('total_value', 'sum'),
    total_profit=('line_profit', 'sum')
).reset_index().sort_values(by='total_value', ascending=False)

px.bar(region, x='total_value', y='city', color='total_profit', color_continuous_scale='emrld',
       title='Revenue by City (colored by profit)')